In [0]:
#creating fraud detection table first strt with reading silver
transactions_silver = spark.table(
    "banking_catalog.silver.transactions"
)

display(transactions_silver)

transaction_id,account_id,transaction_date,amount,transaction_type,merchant
T007,A1005,2025-01-06,70000,Debit,CarDealer
T004,A1003,2025-01-03,4000,Debit,Flipkart
T001,A1001,2025-01-01,3000,Debit,Amazon
T005,A1004,2025-01-04,50000,Debit,Apple
T006,A1005,2025-01-05,25000,Credit,Bonus
T003,A1002,2025-01-01,2500,Debit,Swiggy
T002,A1001,2025-01-02,15000,Credit,Salary


In [0]:
#importing functions which we need 
from pyspark.sql.functions import when, lit

In [0]:
#staring with fraud alerts
fraud_alerts = (
    transactions_silver
    .filter(
        transactions_silver.amount > 50000
    )
    .withColumn(
        "fraud_reason",
        lit("High Value Transaction")
    )
)

In [0]:
display(fraud_alerts)

transaction_id,account_id,transaction_date,amount,transaction_type,merchant,fraud_reason
T007,A1005,2025-01-06,70000,Debit,CarDealer,High Value Transaction


In [0]:
#saving fraud table
fraud_alerts.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "banking_catalog.gold.fraud_alerts"
    )

In [0]:
print(
    "Total Fraud Alerts:",
    fraud_alerts.count()
)

Total Fraud Alerts: 1


Part 2: Risk Categorization

Currently:

Fraud
or
Not Fraud

Real banks usually classify risk levels.

Business Rule
Low Risk
amount <= 20000
Medium Risk
20001 - 50000
High Risk
> 50000

In [0]:
from pyspark.sql.functions import when

In [0]:
#creating risk levels
risk_transactions = (
    transactions_silver
    .withColumn(
        "risk_category",
        when(
            transactions_silver.amount > 50000,
            "High Risk"
        )
        .when(
            transactions_silver.amount > 20000,
            "Medium Risk"
        )
        .otherwise(
            "Low Risk"
        )
    )
)

display(risk_transactions)

transaction_id,account_id,transaction_date,amount,transaction_type,merchant,risk_category
T007,A1005,2025-01-06,70000,Debit,CarDealer,High Risk
T004,A1003,2025-01-03,4000,Debit,Flipkart,Low Risk
T001,A1001,2025-01-01,3000,Debit,Amazon,Low Risk
T005,A1004,2025-01-04,50000,Debit,Apple,Medium Risk
T006,A1005,2025-01-05,25000,Credit,Bonus,Medium Risk
T003,A1002,2025-01-01,2500,Debit,Swiggy,Low Risk
T002,A1001,2025-01-02,15000,Credit,Salary,Low Risk


In [0]:
risk_transactions.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "banking_catalog.gold.transaction_risk_analysis"
    )

Fraud Summary KPI Table
Business Questions
How many transactions occurred?

How many fraud alerts were generated?

What percentage of transactions were flagged?

How many High Risk transactions exist?

In [0]:
#fraud matrics
total_transactions = transactions_silver.count()

total_fraud_alerts = fraud_alerts.count()

fraud_percentage = (
    total_fraud_alerts / total_transactions
) * 100

print("Total Transactions:", total_transactions)
print("Fraud Alerts:", total_fraud_alerts)
print("Fraud Percentage:", round(fraud_percentage,2))

Total Transactions: 7
Fraud Alerts: 1
Fraud Percentage: 14.29


In [0]:
#now lets create fraud summary table
from pyspark.sql import Row

fraud_summary = spark.createDataFrame([

    Row(
        total_transactions=total_transactions,
        total_fraud_alerts=total_fraud_alerts,
        fraud_percentage=round(fraud_percentage,2)
    )

])

display(fraud_summary)

total_transactions,total_fraud_alerts,fraud_percentage
7,1,14.29


In [0]:
#saving fraud Gold table
fraud_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "banking_catalog.gold.fraud_summary"
    )